# F²-FixMatch & Baselines on UCF-101 (Google Colab Runner)

This notebook provides a complete workflow to run training and official 30-view inference for **$F^2$-FixMatch** on Google Colab (16GB T4 GPU).

### Instructions:
1. Go to **Runtime -> Change runtime type** and set Hardware Accelerator to **T4 GPU** (or A100).
2. Run the cells in sequential order.

## Step 1: Clone Repository & Install Dependencies

In [ ]:
# Clone the repository
!git clone https://github.com/MankiratSingh-git/P24-CV.git /content/P24-CV
%cd /content/P24-CV

# Install requirements
!pip install -r requirements.txt

# Verify GPU hardware
!python verify_pipeline.py

## Step 2: Download & Extract UCF-101 (Takes ~45s on Colab Backbone)

In [ ]:
# Download full UCF-101 directly to Colab fast local SSD
!python download_ucf101.py --dest data/ucf101

## Step 3: Mount Google Drive (To Save Checkpoints & Metrics Automatically)

In [ ]:
from google.colab import drive
import os

drive.mount("/content/drive")
GDRIVE_DIR = "/content/drive/MyDrive/P24-CV/checkpoints"
os.makedirs(GDRIVE_DIR, exist_ok=True)
print(f"Checkpoints will be safely persisted in: {GDRIVE_DIR}")

## Step 4: Training Pipelines (Choose which experiment to run)

Because Colab has 16GB VRAM, we can run with `--accum-steps 2` or `4` to match the paper's 2-GPU accumulated mini-batch size.

### Option A: Supervised Baseline (10% Labels Only)

In [ ]:
!python train_supervised_baseline.py \
    --data-root data/ucf101 \
    --splits-dir data/splits \
    --labeled-percent 10 \
    --split 1 \
    --batch-size 8 \
    --accum-steps 2 \
    --epochs 120 \
    --lr 0.2 \
    --output-dir /content/drive/MyDrive/P24-CV/checkpoints/supervised_baseline

### Option B: FixMatch Baseline (RGB Only, 10% Labeled + 90% Unlabeled)

In [ ]:
!python train_fixmatch_baseline.py \
    --data-root data/ucf101 \
    --splits-dir data/splits \
    --labeled-percent 10 \
    --split 1 \
    --batch-size 4 \
    --mu 4 \
    --accum-steps 2 \
    --threshold 0.3 \
    --warmup-epochs 40 \
    --epochs 220 \
    --lr 0.2 \
    --output-dir /content/drive/MyDrive/P24-CV/checkpoints/fixmatch_baseline

### Option C: Full F²-FixMatch (Target Model: RGB + TG, P-Aug, MPL - Paper Target: 71.1%)

In [ ]:
!python train_f2_fixmatch.py \
    --data-root data/ucf101 \
    --splits-dir data/splits \
    --labeled-percent 10 \
    --split 1 \
    --batch-size 4 \
    --mu 4 \
    --accum-steps 2 \
    --threshold 0.3 \
    --alpha 2.0 \
    --beta 2.0 \
    --gamma 1.0 \
    --warmup-epochs 40 \
    --epochs 220 \
    --lr 0.2 \
    --output-dir /content/drive/MyDrive/P24-CV/checkpoints/f2_fixmatch

## Step 5: Official 30-View Test Evaluation (10 Clips x 3 Spatial Crops @ 256x256)

In [ ]:
# Evaluate any checkpoint (change the checkpoint path as needed)
CHECKPOINT = "/content/drive/MyDrive/P24-CV/checkpoints/f2_fixmatch/best_model.pth"

!python evaluate.py \
    --checkpoint "$CHECKPOINT" \
    --data-root data/ucf101 \
    --splits-dir data/splits \
    --split 1 \
    --eval-mode full \
    --crop-size 256